In [6]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Import Libraries

In [7]:
import os

import pandas as pd
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from dataset import *
from model import *
from trainer import Trainer
import random

torch.manual_seed(42)

In [8]:
PATH = "../"
MAX_LEN = 128
BATCH_SIZE = 64

# Loading data

In [9]:
train_data = pd.read_csv(os.path.join(PATH, "train.csv"))
test_data = pd.read_csv(os.path.join(PATH, "test.csv"))

train_data.head()

,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


# Label encoding

In [10]:
le = LabelEncoder()

train_data.rate = le.fit_transform(train_data.rate)
train_data.head()

,rate,text
0,3,Очень понравилось. Были в начале марта с соба...
1,4,В целом магазин устраивает.\nАссортимент позво...
2,4,"Очень хорошо что открылась 5 ка, теперь не над..."
3,2,Пятёрочка громко объявила о том как она заботи...
4,2,"Тесно, вечная сутолока, между рядами трудно ра..."


In [11]:
import spacy
nlp = spacy.load("ru_core_news_sm")

In [12]:
train_data['text'] = train_data['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        #not token.is_stop
        #not token.is_punct
        not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [13]:
test_data['text'] = test_data['text'].apply(
    lambda x: ' '.join(
        token.lemma_.lower() for token in nlp(x) if
        #not token.is_stop
        #not token.is_punct
        not token.is_digit
        and not token.like_email
        and not token.like_num
        and not token.is_space
    )
)

In [14]:
import gensim.downloader as api
from gensim.models import KeyedVectors
from pymorphy3 import MorphAnalyzer
w2v_model = KeyedVectors.load_word2vec_format('~/gensim-data/ruwikiruscorpora-nobigrams_upos_skipgram_300_5_2018.vec.gz', binary=False,encoding='utf-8')


In [15]:
def get_synonyms(word, topn=5):
    try:
        return w2v_model.most_similar(word, topn=topn)
    except KeyError:
        return []
def get_syn (word) :
    morph = MorphAnalyzer()
    normal_form = morph.parse(word)
    syn  = normal_form[0][2] + '_' + str(normal_form[0][1]).split(',')[0]
    synonyms = get_synonyms(syn)
    if synonyms == [] :
        return None
    syn = synonyms[0][0].split('_')[0]
    return syn
def aug_sent(sentence, p = 0.8) :
    for i in range(len(sentence)) :
        r = random.random()
        if r < p :                   
            syn = get_syn(sentence[i])
            if syn != None :
                sentence[i] = syn
    return sentence

# Train Test split

In [16]:
train_split, val_split = train_test_split(train_data, test_size=0.15, random_state=42)

In [17]:
train_1 = train_split[train_split['rate'] == 1]

In [ ]:
sent_list = []
rate_list = []
for index, row in train_1.iterrows():
    sent = row["text"].split()
    new_sent = aug_sent(sent, p= 0.8)
    sent_str = ' '.join(new_sent)
    sent_list.append(sent_str)
    rate_list.append(1)   

In [ ]:
new_rows = pd.DataFrame({'text': sent_list, 'rate': rate_list })
# Добавляем к исходному DataFrame
train_split = pd.concat([train_split, new_rows], ignore_index=True)

In [15]:
train_2 = train_split[train_split['rate'] == 2]
sent_list = []
rate_list = []
for index, row in train_2.iterrows():
    sent = row["text"].split()
    new_sent = aug_sent(sent, p= 0.8)
    sent_str = ' '.join(new_sent)
    sent_list.append(sent_str)
    rate_list.append(2) 
new_rows = pd.DataFrame({'text': sent_list, 'rate': rate_list })
train_split = pd.concat([train_split, new_rows], ignore_index=True)

In [16]:
train_3 = train_split[train_split['rate'] == 3]
sent_list = []
rate_list = []
for index, row in train_3.iterrows():
    sent = row["text"].split()
    new_sent = aug_sent(sent, p= 0.8)
    sent_str = ' '.join(new_sent)
    sent_list.append(sent_str)
    rate_list.append(3) 
new_rows = pd.DataFrame({'text': sent_list, 'rate': rate_list })
train_split = pd.concat([train_split, new_rows], ignore_index=True)

# Loading tokenizer from pretrained

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "cointegrated/rubert-tiny2", truncation=True, do_lower_case=True)

# Creating datasets and dataloaders

In [ ]:
train_dataset = FiveDataset(train_split, tokenizer, MAX_LEN)
val_dataset = FiveDataset(val_split, tokenizer, MAX_LEN)
test_dataset = FiveDataset(test_data, tokenizer, MAX_LEN)

In [ ]:
train_params = {"batch_size": BATCH_SIZE,
                "shuffle": True,
                "num_workers": 0
                }

test_params = {"batch_size": BATCH_SIZE,
               "shuffle": False,
               "num_workers": 0
               }

train_dataloader = DataLoader(train_dataset, **train_params)
val_dataloader = DataLoader(val_dataset, **test_params)
test_dataloader = DataLoader(test_dataset, **test_params)

# Loading pretrained model from Huggingface

In [ ]:
config = {
    "num_classes": 5,
    "dropout_rate": 0.1
}
model = ModelForClassification(
    "cointegrated/rubert-tiny2",
    config=config
)

# Creating Trainer object and fitting the model

In [ ]:
trainer_config = {
    "lr": 3e-4,
    "n_epochs": 1,
    "weight_decay": 1e-6,
    "batch_size": BATCH_SIZE,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,
}
t = Trainer(trainer_config)

In [ ]:
t.fit(
    model,
    train_dataloader,
    val_dataloader
)

In [ ]:
predictions = t.predict(val_dataloader)

In [ ]:
print(classification_report(val_split['rate'], predictions))

# Save model

In [ ]:
t.save("baseline_model.ckpt")

# Load pretrained Model

In [ ]:
t = Trainer.load("baseline_model.ckpt")

# Get testset predictions


In [ ]:
predictions = t.predict(test_dataloader)

# Create submission


In [ ]:
sample_submission = pd.read_csv(os.path.join(PATH, "sample_submission.csv"))
sample_submission["rate"] = predictions
sample_submission.rate = le.inverse_transform(sample_submission.rate)
sample_submission.head()

In [ ]:
sample_submission.to_csv("submission.csv", index=False)